# Prueba 1: ¿hasta dónde llega el efecto electrónico de la glicina en GluN1?

Geometría congelada (1PB7 con hidrógenos, preparada en local). PM7, MOZYME, `1SCF GRADIENTS`,
`SCFCRT=2.D-6 RELTHR=0.01` (con el umbral normal de MOZYME la densidad no baja de ~1,5·10⁻⁵).

| Sistema | Contenido |
|---|---|
| A | proteína + glicina |
| B | proteína sola |
| C | proteína + la glicina como cargas puntuales (sus cargas ESP de PM7, `MM_CHARGES`) |
| C2 | proteína + la glicina como las cargas puntuales que tiene dentro de la proteína (de A) |
| D | glicina sola (cargas ESP) |

A − B es todo el efecto de la glicina sobre las cargas y fuerzas de la proteína; C − B, lo que da un campo clásico; A − C,
lo que la glicina cuántica añade; A − C2, lo mismo sin su polarización por la proteína. En CPU (local), A − C2 lejos de la
glicina salió ~0,6 veces C2 − B a toda distancia; estas corridas miran si eso depende de cómo MOZYME trata el largo
alcance (`CUTOFF=25`; con el valor por defecto, más allá de ~10 Å usa cargas y dipolos) o del vacío (`EPS=78.4`). Ruido:
dos corridas iguales (`r2`, lo que no repite la GPU) y los átomos en orden inverso (`rev`). También `CUTOFF=15` y el SCF
convencional sin orbitales localizados.

| Bloque | Tiempo |
|---|---|
| Compilar MOPAC | ~10 min con los 12 núcleos de una sesión A100 |
| GPU: hasta 20 corridas de MOZYME, ~80 iteraciones cada una | sin medir; en CPU cada una tarda 35–95 min |
| SCF convencional, 4 614 átomos | hasta `CONV_TIMEOUT` por corrida |

Las corridas van en orden de importancia; si el tiempo se acaba, lo terminado ya sirve.

**GPU.** Este cuaderno pide una A100: si Colab asigna otra (ha pasado que dé una L4), se detiene en la
celda 2 sin compilar nada. Para otra GPU use `allo_t1_colab.ipynb` o ponga `REQUIRE_GPU = None`. Mire cuánto tarda la
primera corrida (`A_vac`, línea `"seconds"`): este sistema no se ha medido en GPU.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU A100*, y luego *Ejecutar todo*. Los
resultados se guardan en Google Drive (`MyDrive/mopac_paper_bench/allo_t1b_a100`); si la sesión se corta, vuelva a
ejecutar todo y lo terminado se salta. La última celda descarga un zip.

## 1. Configuración

In [ ]:
COMMIT = '323b6076'
RESULTS_NAME = 'allo_t1b'      # la carpeta lleva además el nombre de la GPU: allo_t1b_t4, allo_t1b_a100
USE_DRIVE = True
REQUIRE_GPU = 'A100'           # 'A100' o 'T4': parar si la sesión tiene otra GPU; None: cualquiera
CONV_TIMEOUT = 2700            # s por corrida del SCF convencional

## 2. GPU y carpeta de resultados

In [ ]:
import os, re, subprocess, sys, time, shutil, json, gzip, base64, threading
from pathlib import Path

def run(cmd, check=True, env=None):
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

run('nvidia-smi')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
NCPU = os.cpu_count()
CC = gpu.split(',')[-1].strip().replace('.', '')
if REQUIRE_GPU and REQUIRE_GPU not in gpu:
    raise RuntimeError(f'Esta sesión tiene {gpu}, no una {REQUIRE_GPU}.')
GPU_TAG = [w for w in gpu.split(',')[0].split() if w.lower() not in ('nvidia', 'tesla')][0].split('-')[0].lower()
if not RESULTS_NAME.endswith(f'_{GPU_TAG}'):
    RESULTS_NAME = f'{RESULTS_NAME}_{GPU_TAG}'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/mopac_paper_bench')
else:
    BASE = Path('/content/mopac_paper_bench')
OUT = BASE / RESULTS_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'env.json').write_text(json.dumps({'gpu': gpu, 'ncpu': NCPU, 'commit': COMMIT}, indent=1))
print('resultados en', OUT)

## 3. Compilar MOPAC

~10 minutos.

In [ ]:
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
MOPAC = BUILD / 'mopac'
if not MOPAC.exists():
    run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
        'libopenblas-dev liblapack-dev > /dev/null')
    os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
    shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(BUILD, ignore_errors=True)
    run(f'git clone -q https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
    run(f'git -C {SRC} checkout -q {COMMIT}')
    head = subprocess.run(f'git -C {SRC} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
    if not head.startswith(COMMIT):
        raise SystemExit(f'HEAD {head} no es COMMIT {COMMIT}')
    stamp = OUT / 'commit.txt'
    if stamp.exists() and not head.startswith(stamp.read_text().strip()[:8]):
        raise SystemExit(f'{OUT} tiene resultados de {stamp.read_text().strip()}; use otro RESULTS_NAME')
    stamp.write_text(head + '\n')
    t0 = time.time()
    run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
        f'-DENABLE_MKL=OFF -DBLA_VENDOR=OpenBLAS > /content/cmake.log 2>&1 || (tail -40 /content/cmake.log; exit 1)')
    run(f'cmake --build {BUILD} --target mopac --parallel {NCPU} > /content/build.log 2>&1 '
        f'|| (grep -i -B2 -A6 error /content/build.log | head -80; exit 1)')
    print(f'compilado en {time.time() - t0:.0f} s')
run(f'ls -la {MOPAC}')

## 4. Entradas (estructura preparada en local y scripts)

In [ ]:
Path('/content/t1_tools.py').write_text('"""Test 1: how far the glycine changes the protein\'s electrons, at a frozen geometry (MOZYME, PM7).\n\n    python t1_tools.py decks ATOMS_CSV WORK_DIR CHARGE      write the decks (A, B, D; C after D)\n    python t1_tools.py mmcharges WORK_DIR                   C\'s point charges from D\'s ESP charges\n    python t1_tools.py c2 WORK_DIR VARIANT                  C2_VARIANT: the glycine\'s charges in A_VARIANT\n    python t1_tools.py analyse WORK_DIR OUT_DIR             tables and figures\n\nA = protein + glycine, B = protein alone, C = protein + the glycine as PM7 ESP point charges (MM_CHARGES),\nD = glycine alone (ESP charges). Every protein atom keeps its place and its index in A, B and C, so\nA - B is the whole effect of the glycine on the protein\'s charges and forces, C - B the part a fixed set of\npoint charges gives (the classical field), A - C what the quantum glycine adds (charge transfer, exchange,\npolarisation of the glycine, its true charge distribution). C2 = the glycine as the point charges it has inside\nthe protein (from A), so A - C2 leaves out its polarisation by the protein. The noise: the same jobs repeated (GPU)\nand with the atoms in reverse order (another path through the SCF to the same answer).\n"""\nimport csv\nimport json\nimport re\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\n# MOZYME\'s density stops improving at ~1.5e-5 with its default LMO threshold (1e-13); RELTHR=0.01 (1e-15) takes it\n# to ~2e-6, where the energy wanders by a few 1e-6 kcal/mol between iterations, so SCFCRT=1e-7 never converges.\n# XS, XP, XW: no overlap, density or M.O. matrices in the AUX file (1.7 GB per run for this protein; not used here).\nCOMMON = "PM7 1SCF GRADIENTS GEO-OK AUX(PRECISION=4,XS,XP,XW) SCFCRT=2.D-6 RELTHR=0.01 ITRY=300"\n\n\ndef load(atoms_csv):\n    rows = list(csv.DictReader(open(atoms_csv)))\n    prot = [(r["element"], np.array([float(r[k]) for k in "xyz"])) for r in rows if r["resname"] != "LIG"]\n    lig = [(r["element"], np.array([float(r[k]) for k in "xyz"])) for r in rows if r["resname"] == "LIG"]\n    meta = [(r["resname"], r["resid"], r["atom"], r["kind"]) for r in rows if r["resname"] != "LIG"]\n    return prot, lig, meta\n\n\ndef deck(atoms, kw, title):\n    lines = [kw, title, ""]\n    lines += [f"{el:2s} {x[0]:12.6f} 1 {x[1]:12.6f} 1 {x[2]:12.6f} 1" for el, x in atoms]\n    return "\\n".join(lines) + "\\n"\n\n\nVARIANTS = {          # name: (extra keywords, environment, MOZYME?)\n    "vac": ("", {}, True), "r2": ("", {}, True), "cpu": ("", {"MOPAC_NOGPU": "1"}, True),\n    "rev": ("", {}, True),      # atoms in reverse order: same physics, another path through the SCF\n    "c15": ("CUTOFF=15", {}, True), "c25": ("CUTOFF=25", {}, True), "eps": ("EPS=78.4", {}, True),\n    "conv": ("", {}, False),\n    # CUTOFF=15 and 25 failed the SCF at SCFCRT=2.D-6 (L4, 300 iterations): the same at 1.D-5, with its own base\n    # and repeat, the GPU SCF trace of one failing job, and the CPU path of the same job\n    "vacl": ("SCFCRT=1.D-5", {}, True), "vacl2": ("SCFCRT=1.D-5", {}, True),\n    "c15l": ("CUTOFF=15 SCFCRT=1.D-5", {}, True), "c25l": ("CUTOFF=25 SCFCRT=1.D-5", {}, True),\n    "c15t": ("CUTOFF=15", {"MOPAC_MOZYME_SCF_TRACE": "1"}, True),\n    "c15cpu": ("CUTOFF=15 PL", {"MOPAC_NOGPU": "1"}, True),\n}\n\n\ndef keywords(kw):\n    """COMMON with the variant\'s keywords; a keyword=value already in COMMON is replaced, not repeated."""\n    base, extra = COMMON, []\n    for tok in kw.split():\n        key = tok.split("=")[0] + "="\n        if "=" in tok and re.search(rf"(?<!\\S){re.escape(key)}", base):\n            base = re.sub(rf"(?<!\\S){re.escape(key)}\\S+", tok, base)\n        else:\n            extra.append(tok)\n    return base, " ".join(extra)\n\n\ndef write_decks(atoms_csv, wd, charge):\n    prot, lig, meta = load(atoms_csv)\n    wd.mkdir(parents=True, exist_ok=True)\n    (wd / "meta.json").write_text(json.dumps({"meta": meta, "lig": [x.tolist() for _, x in lig],\n                                              "prot": [x.tolist() for _, x in prot]}))\n    (wd / "D").mkdir(exist_ok=True)\n    (wd / "D" / "job.mop").write_text(deck(lig, "PM7 1SCF CHARGE=0 ESP GEO-OK AUX(PRECISION=4)", "glycine alone"))\n    for v, (kw, env, mozyme) in VARIANTS.items():\n        for sysname, atoms in (("A", prot + lig), ("B", prot), ("C", prot)):\n            if v == "rev":\n                atoms = atoms[::-1]\n            d = wd / f"{sysname}_{v}"\n            d.mkdir(exist_ok=True)\n            base, kws = keywords(kw)\n            extra = (" MOZYME" if mozyme else "") + (f" {kws}" if kws else "")\n            if sysname == "C":\n                extra += \' MM_CHARGES="glycine.chg"\' if mozyme else " QMMM"\n            (d / "job.mop").write_text(deck(atoms, f"{base} CHARGE={charge}{extra}", f"{sysname} {v}"))\n            (d / "env.json").write_text(json.dumps(env))\n\n\ndef aux_block(text, key):\n    m = re.search(r"(?<![A-Z_])" + re.escape(key) + r"(?::[A-Z/]+)?\\[(\\d+)\\]=", text)\n    if not m:\n        return None\n    n, vals = int(m.group(1)), []\n    for ln in text[m.end():].splitlines():\n        if ln.strip().startswith("#"):\n            continue\n        for tok in ln.split():\n            vals.append(float(tok.replace("D", "E")))\n            if len(vals) == n:\n                return np.array(vals)\n    return None\n\n\ndef write_mmcharges(wd):\n    meta = json.loads((wd / "meta.json").read_text())\n    q = aux_block((wd / "D" / "job.aux").read_text(errors="ignore"), "ELECTROSTATIC_POTENTIAL_CHARGES")\n    lig = np.array(meta["lig"])\n    prot = np.array(meta["prot"])\n    text = "".join(f"{x[0]:.6f} {x[1]:.6f} {x[2]:.6f} {c:.6f}\\n" for x, c in zip(lig, q))\n    pot = [332.0637 * float((q / np.linalg.norm(lig - x, axis=1)).sum()) for x in prot]\n    molin = f"glycine ESP charges, potential at each protein atom (kcal/mol, +1 charge)\\n{len(prot)} 0\\n" + \\\n        "".join(f"x y z q {v:.6f}\\n" for v in pot)\n    for d in wd.glob("C_*"):\n        (d / "glycine.chg").write_text(text)\n        (d / "mol.in").write_text(molin)\n    return q\n\n\ndef write_c2(wd, v):\n    """C2_v: deck of C_v, the glycine\'s charges taken from A_v (its last atoms)."""\n    lig = np.array(json.loads((wd / "meta.json").read_text())["lig"])\n    q = aux_block((wd / f"A_{v}" / "job.aux").read_text(errors="ignore"), "ATOM_CHARGES")\n    if q is None:\n        raise SystemExit(f"A_{v}/job.aux has no ATOM_CHARGES (see A_{v}/job.out); C2_{v} not written")\n    q = q[-len(lig):]\n    d = wd / f"C2_{v}"\n    d.mkdir(exist_ok=True)\n    lines = (wd / f"C_{v}" / "job.mop").read_text().splitlines()\n    lines[1] = f"C2 {v}: the glycine as the point charges it has in A"\n    (d / "job.mop").write_text("\\n".join(lines) + "\\n")\n    (d / "env.json").write_text((wd / f"C_{v}" / "env.json").read_text())\n    (d / "glycine.chg").write_text("".join(f"{x[0]:.6f} {x[1]:.6f} {x[2]:.6f} {c:.6f}\\n" for x, c in zip(lig, q)))\n    return q\n\n\ndef read_run(d, n):\n    aux = d / "job.aux"\n    if not aux.exists():\n        return None\n    t = aux.read_text(errors="ignore")\n    q = aux_block(t, "ATOM_CHARGES")\n    g = aux_block(t, "GRADIENTS")\n    x = aux_block(t, "ATOM_X_OPT") if aux_block(t, "ATOM_X_OPT") is not None else aux_block(t, "ATOM_X")\n    if q is None:\n        return None\n    g = g.reshape(-1, 3) if g is not None and len(g) == 3 * len(q) else None\n    x = x.reshape(-1, 3) if x is not None else None\n    if d.name.endswith("_rev"):     # back to the input order\n        q, g, x = q[::-1], g[::-1] if g is not None else None, x[::-1] if x is not None else None\n    return {"q": q[:n], "g": g[:n] if g is not None else None, "x": x[:n] if x is not None else None}\n\n\nBINS = [0, 4, 6, 8, 10, 12, 15, 20, 25, 30, 40]\n\n\ndef binned(r, v):\n    out = []\n    for lo, hi in zip(BINS, BINS[1:]):\n        s = v[(r >= lo) & (r < hi)]\n        out.append((lo, hi, len(s), np.median(s) if len(s) else np.nan, np.percentile(s, 95) if len(s) else np.nan,\n                    s.max() if len(s) else np.nan))\n    return out\n\n\ndef analyse(wd, od):\n    od.mkdir(parents=True, exist_ok=True)\n    meta = json.loads((wd / "meta.json").read_text())\n    prot, lig = np.array(meta["prot"]), np.array(meta["lig"])\n    n = len(prot)\n    r = np.linalg.norm(prot[:, None] - lig[None], axis=2).min(1)\n    runs = {}\n    for d in sorted(wd.glob("[ABC]*_*")):\n        res = read_run(d, n)\n        if res is not None:\n            if res["x"] is not None:\n                dev = np.abs(res["x"] - prot).max()\n                if dev > 1e-3:\n                    raise RuntimeError(f"{d.name}: atom order or geometry differs from the input ({dev:.4f} A)")\n            runs[d.name] = res\n    lines = [f"runs read: {\', \'.join(sorted(runs))}", f"protein atoms: {n}; farthest from the glycine: {r.max():.1f} A"]\n    table = []\n\n    def change(a, b, what):         # b may be None (a alone) or a pair (a minus the difference of two runs)\n        if a not in runs or (b is not None and b not in runs):\n            return None\n        v = runs[a][what]\n        return None if v is None else v if b is None else None if runs[b][what] is None else v - runs[b][what]\n\n    def diff(a, b, what):           # a, b: run names, or pairs of run names (the difference of two differences)\n        va = change(*a, what) if isinstance(a, tuple) else change(a, None, what)\n        vb = change(*b, what) if isinstance(b, tuple) else change(b, None, what)\n        if va is None or vb is None:\n            return None\n        return np.abs(va - vb) if what == "q" else np.linalg.norm(va - vb, axis=1)\n\n    v0 = "vac" if "A_vac" in runs else "cpu"       # the base runs: GPU, or CPU when there are no GPU runs\n    A, B, C = f"A_{v0}", f"B_{v0}", f"C_{v0}"\n    lines.append(f"base runs: {v0}")\n    for what, unit in (("q", "e"), ("g", "kcal/mol/A")):\n        lines.append(f"\\n### {\'atomic charges\' if what == \'q\' else \'forces\'} ({unit}): median / 95th percentile / "\n                     f"largest per distance bin")\n        pairs = [("A-B (whole effect)", A, B), ("C-B (point charges)", C, B), ("A-C (beyond point charges)", A, C),\n                 ("C2-B (in-situ point charges)", f"C2_{v0}", B), ("A-C2 (beyond in-situ charges)", A, f"C2_{v0}"),\n                 ("noise A-A", "A_vac", "A_r2"), ("noise B-B", "B_vac", "B_r2"), ("noise C-C", "C_vac", "C_r2"),\n                 ("noise A-A, reversed order", A, "A_rev"), ("noise B-B, reversed order", B, "B_rev"),\n                 ("noise C-C, reversed order", C, "C_rev"),\n                 ("noise of A-B (reversed order)", (A, B), ("A_rev", "B_rev")),\n                 ("noise of C-B (reversed order)", (C, B), ("C_rev", "B_rev")),\n                 ("noise of A-C (reversed order)", (A, C), ("A_rev", "C_rev")),\n                 ("GPU-CPU A", "A_vac", "A_cpu"),\n                 ("A-C, CPU", "A_cpu", "C_cpu"), ("A-B, CPU", "A_cpu", "B_cpu"),\n                 ("A-C, CUTOFF=15", "A_c15", "C_c15"), ("A-C, CUTOFF=25", "A_c25", "C_c25"),\n                 ("A-B, CUTOFF=25", "A_c25", "B_c25"), ("C2-B, CUTOFF=25", "C2_c25", "B_c25"),\n                 ("A-C2, CUTOFF=25", "A_c25", "C2_c25"),\n                 ("A-B, water (COSMO)", "A_eps", "B_eps"),\n                 # MM_CHARGES enter only the core Hamiltonian; COSMO does not screen them, so in water C and C2\n                 # feel the glycine\'s charges unscreened while A\'s glycine is screened\n                 ("A-C, water (C unscreened)", "A_eps", "C_eps"), ("C2-B, water (C2 unscreened)", "C2_eps", "B_eps"),\n                 ("A-C2, water (C2 unscreened)", "A_eps", "C2_eps"),\n                 ("A-B, SCFCRT=1e-5", "A_vacl", "B_vacl"), ("C2-B, SCFCRT=1e-5", "C2_vacl", "B_vacl"),\n                 ("A-C2, SCFCRT=1e-5", "A_vacl", "C2_vacl"),\n                 ("noise A-A, SCFCRT=1e-5", "A_vacl", "A_vacl2"), ("noise B-B, SCFCRT=1e-5", "B_vacl", "B_vacl2"),\n                 ("A: SCFCRT 1e-5 - 2e-6", "A_vacl", "A_vac"),\n                 ("A-B, CUTOFF=15, 1e-5", "A_c15l", "B_c15l"), ("C2-B, CUTOFF=15, 1e-5", "C2_c15l", "B_c15l"),\n                 ("A-C2, CUTOFF=15, 1e-5", "A_c15l", "C2_c15l"),\n                 ("A-B, CUTOFF=25, 1e-5", "A_c25l", "B_c25l"), ("C2-B, CUTOFF=25, 1e-5", "C2_c25l", "B_c25l"),\n                 ("A-C2, CUTOFF=25, 1e-5", "A_c25l", "C2_c25l"),\n                 ("B: CUTOFF=15 - default, 1e-5", "B_c15l", "B_vacl"),\n                 ("B: CUTOFF=15, CPU - GPU", "B_c15cpu", "B_c15t"),\n                 ("A-B, conventional SCF", "A_conv", "B_conv")]\n        if what == "q":\n            pairs.append(("A-C, conventional SCF (C: point potentials)", "A_conv", "C_conv"))\n        hdr = "  " + " " * 44 + "".join(f"{lo:>4d}-{hi:<4d}" + " " * 14 for lo, hi in zip(BINS, BINS[1:]))\n        lines.append("  bins (A from the glycine): " + ", ".join(f"{lo}-{hi}" for lo, hi in zip(BINS, BINS[1:])))\n        for label, a, b in pairs:\n            v = diff(a, b, what)\n            if v is None or (v0 == "cpu" and label.endswith(", CPU")):\n                continue\n            bs = binned(r, v)\n            table += [{"quantity": what, "pair": label, "lo": lo, "hi": hi, "n": k, "median": med, "p95": p95,\n                       "max": mx} for lo, hi, k, med, p95, mx in bs]\n            lines.append(f"  {label:44s} " + " | ".join(f"{med:.1e}/{p95:.1e}/{mx:.1e}" for _, _, _, med, p95, mx in bs))\n        # the question of the CUTOFF runs: is A-C2 still ~0.6 times C2-B far from the glycine?\n        rows = []\n        for label, a, c2, b in (("default cutoff, 2e-6", "A_vac", "C2_vac", "B_vac"),\n                                ("default cutoff, CPU", "A_cpu", "C2_cpu", "B_cpu"),\n                                ("default cutoff, 1e-5", "A_vacl", "C2_vacl", "B_vacl"),\n                                ("CUTOFF=15, 1e-5", "A_c15l", "C2_c15l", "B_c15l"),\n                                ("CUTOFF=25, 1e-5", "A_c25l", "C2_c25l", "B_c25l")):\n            v1, v2 = diff(a, c2, what), diff(c2, b, what)\n            if v1 is None or v2 is None:\n                continue\n            rows.append(f"  {label:44s} " + " | ".join(f"{m1 / m2:6.2f}" for (_, _, _, m1, _, _), (_, _, _, m2, _, _)\n                                                        in zip(binned(r, v1), binned(r, v2))))\n        if rows:\n            lines.append(f"  median(A-C2) / median(C2-B) per bin:")\n            lines += rows\n    with open(od / "binned.csv", "w", newline="") as f:\n        w = csv.DictWriter(f, fieldnames=["quantity", "pair", "lo", "hi", "n", "median", "p95", "max"])\n        w.writeheader()\n        w.writerows(table)\n    # residues: largest |A-C| charge change per residue against its distance; the aromatic path singled out\n    if A in runs and C in runs:\n        dq = np.abs(runs[A]["q"] - runs[C]["q"])\n        noise = diff((A, C), ("A_rev", "C_rev"), "q")\n        res = {}\n        for i, (rn, rid, an, kind) in enumerate(meta["meta"]):\n            k = f"{rn}{rid}"\n            res.setdefault(k, []).append(i)\n        path = {"PHE92", "TRP223", "PHE221", "TYR203", "HIS201", "HIS197", "TYR173", "PHE171"}\n        lines.append("\\n### residues: largest |A-C| charge change (e) and distance of the residue from the glycine (A)")\n        rows = sorted(((dq[ix].max(), r[ix].min(), k) for k, ix in res.items()), key=lambda t: t[1])\n        for lo, hi in zip(BINS, BINS[1:]):\n            sel = [t for t in rows if lo <= t[1] < hi]\n            if not sel:\n                continue\n            top = max(sel)\n            pth = [f"{k}:{m:.1e}" for m, d_, k in sel if k in path]\n            lines.append(f"  {lo:>2d}-{hi:<2d} A: {len(sel):3d} residues, median {np.median([t[0] for t in sel]):.1e}, "\n                         f"largest {top[0]:.1e} ({top[2]})" + (f"; path residues {\', \'.join(pth)}" if pth else ""))\n        if noise is not None:\n            lines.append(f"  noise of A-C (atoms in reverse order): median {np.median(noise):.1e}, 99th percentile "\n                         f"{np.percentile(noise, 99):.1e}, largest {noise.max():.1e}")\n        # is the aromatic path more affected than other residues as far from the glycine (within 2 A)?\n        lines.append("\\n### path residues: percent of residues at the same distance (+-2 A) with a smaller largest change")\n        for label, a, b in (("A-B", A, B), ("A-C", A, C), ("A-C2", A, f"C2_{v0}")):\n            v = diff(a, b, "q")\n            if v is None:\n                continue\n            per = {k: (v[ix].max(), r[ix].min()) for k, ix in res.items()}\n            ranks = []\n            for k in sorted(path, key=lambda k: per[k][1]):\n                m, d_ = per[k]\n                peers = np.array([w for w, dd in per.values() if abs(dd - d_) < 2.0])\n                ranks.append(f"{k} ({d_:.1f} A) {100 * np.mean(peers < m):.0f}")\n            lines.append(f"  {label:5s} " + ", ".join(ranks))\n        try:\n            import matplotlib\n            matplotlib.use("Agg")\n            import matplotlib.pyplot as plt\n            fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))\n            for ax, what, unit in ((axs[0], "q", "e"), (axs[1], "g", "kcal/mol/Å")):\n                for label, a, b, c in (("A−B", A, B, "#1f5fa8"), ("C−B", C, B, "#9a9a9a"), ("A−C", A, C, "#c2452d"),\n                                       ("noise of A−C", (A, C), ("A_rev", "C_rev"), "#2b8a3e")):\n                    v = diff(a, b, what)\n                    if v is not None:\n                        ax.scatter(r, np.maximum(v, 1e-9), s=2, alpha=0.35, color=c, label=label)\n                ax.set_yscale("log")\n                ax.set_xlabel("distance from the glycine (Å)")\n                ax.set_ylabel(f"|change| ({unit})")\n                ax.legend(markerscale=5, fontsize=8)\n            fig.tight_layout()\n            fig.savefig(od / "decay.png", dpi=150)\n        except Exception as e:      # the figure is optional\n            lines.append(f"(no figure: {e})")\n    (od / "summary.txt").write_text("\\n".join(lines) + "\\n")\n    print("\\n".join(lines))\n\n\nif __name__ == "__main__":\n    cmd = sys.argv[1]\n    if cmd == "decks":\n        write_decks(Path(sys.argv[2]), Path(sys.argv[3]), int(sys.argv[4]))\n    elif cmd == "mmcharges":\n        print(write_mmcharges(Path(sys.argv[2])))\n    elif cmd == "c2":\n        print(write_c2(Path(sys.argv[2]), sys.argv[3]))\n    elif cmd == "analyse":\n        analyse(Path(sys.argv[2]), Path(sys.argv[3]))\n')
Path('/content/run_t1.py').write_text('"""Run test-1 MOPAC jobs: python run_t1.py MOPAC WORK_DIR JOB[,JOB...] [PARALLEL] [TIMEOUT_S]\n\nEach job directory holds job.mop and env.json (extra environment); finished jobs (job.aux present and a\nstatus.json) are skipped. job.log: MOPAC\'s screen output. status.json: wall time, exit code, SCF status lines, MOPAC\'s error messages.\n"""\nimport json\nimport os\nimport re\nimport subprocess\nimport sys\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\n\ndef messages(out):\n    """The messages MOPAC lists in the box at the end of job.out when a job did not end normally."""\n    i = out.rfind("Error and normal termination messages")\n    if i < 0:\n        return []\n    msgs = []\n    for ln in out[i:].splitlines()[1:]:\n        s = ln.strip().strip("*").strip()\n        if s.startswith("JOB ENDED NORMALLY"):\n            break\n        if s:\n            msgs.append(s)\n    return msgs[:5]\n\n\ndef run_job(mopac, d, timeout):\n    st = d / "status.json"\n    if st.exists() and (d / "job.aux").exists():\n        res = json.loads(st.read_text())\n        if res.get("heat") is None and (d / "job.out").exists():\n            res["errors"] = messages((d / "job.out").read_text(errors="ignore")) or res["errors"]\n            st.write_text(json.dumps(res, indent=1))\n        return res\n    env = dict(os.environ)\n    if (d / "env.json").exists():\n        env.update(json.loads((d / "env.json").read_text()))\n    t0 = time.time()\n    try:\n        p = subprocess.run([mopac, "job.mop"], cwd=d, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                           text=True, timeout=timeout)\n        code, log = p.returncode, p.stdout\n    except subprocess.TimeoutExpired:\n        code, log = "timeout", ""\n    (d / "job.log").write_text(log)\n    out = (d / "job.out").read_text(errors="ignore") if (d / "job.out").exists() else ""\n    res = {"job": d.name, "seconds": round(time.time() - t0, 1), "exit": code,\n           "heat": (re.findall(r"FINAL HEAT OF FORMATION =\\s+(-?\\d+\\.\\d+)", out) or [None])[-1],\n           "scf": re.findall(r"\\[MOZYME GPU SCF\\] status=\\S+.*", log + out)[-3:],\n           "errors": messages(out) or [ln.strip() for ln in out.splitlines() if "ERROR" in ln.upper()][:5]}\n    st.write_text(json.dumps(res, indent=1))\n    return res\n\n\nif __name__ == "__main__":\n    mopac, wd = sys.argv[1], Path(sys.argv[2])\n    jobs = [wd / j for j in sys.argv[3].split(",")]\n    par = int(sys.argv[4]) if len(sys.argv) > 4 else 1\n    timeout = float(sys.argv[5]) if len(sys.argv) > 5 else None\n    with ThreadPoolExecutor(par) as ex:\n        for r in ex.map(lambda d: run_job(mopac, d, timeout), jobs):\n            print(json.dumps(r), flush=True)\n')

Path('/content/1PB7_atoms.csv').write_bytes(gzip.decompress(base64.b64decode('...')))
WD = OUT / 'runs'
if not (WD / 'meta.json').exists():
    run(f'{sys.executable} /content/t1_tools.py decks /content/1PB7_atoms.csv {WD} 2')
run(f'{sys.executable} /content/run_t1.py {MOPAC} {WD} D')
run(f'{sys.executable} /content/t1_tools.py mmcharges {WD}')

## 5. Corridas en GPU

Una tras otra, en orden de importancia. C2 sale de las cargas de A de la misma variante, así que va después y solo si A
terminó con calor de formación. Si una corrida falla, su línea muestra los mensajes de MOPAC (`"errors"`) y el
cuaderno sigue con el bloque siguiente. El análisis se rehace después de cada bloque.

In [ ]:
blocks = [('vac', ['A_vac', 'B_vac', 'C_vac', 'C2_vac']), ('c25', ['A_c25', 'B_c25', 'C_c25', 'C2_c25']),
          ('eps', ['A_eps', 'B_eps', 'C_eps', 'C2_eps']), ('r2', ['A_r2', 'B_r2', 'C_r2']),
          ('rev', ['A_rev', 'B_rev', 'C_rev']), ('c15', ['A_c15', 'B_c15', 'C_c15'])]
for v, jobs in blocks:
    first = [j for j in jobs if not j.startswith('C2_')]
    run(f'{sys.executable} /content/run_t1.py {MOPAC} {WD} {",".join(first)} 1', check=False)
    st = WD / f'A_{v}' / 'status.json'
    if len(first) < len(jobs) and st.exists() and json.loads(st.read_text()).get('heat'):
        if run(f'{sys.executable} /content/t1_tools.py c2 {WD} {v}', check=False) == 0:
            run(f'{sys.executable} /content/run_t1.py {MOPAC} {WD} C2_{v} 1', check=False)
    run(f'{sys.executable} /content/t1_tools.py analyse {WD} {OUT / "analysis"}', check=False)

## 6. SCF convencional (sin orbitales localizados)

Control de MOZYME: el mismo A − B con la matriz completa. Hasta `CONV_TIMEOUT` segundos por corrida; si no termina, el
resto de los resultados vale igual.

In [ ]:
run(f'{sys.executable} /content/run_t1.py {MOPAC} {WD} A_conv,B_conv,C_conv 1 {CONV_TIMEOUT}', check=False)
run(f'{sys.executable} /content/t1_tools.py analyse {WD} {OUT / "analysis"}')

## 7. Figura y descarga

In [ ]:
from IPython.display import Image, display
if (OUT / 'analysis' / 'decay.png').exists():
    display(Image(str(OUT / 'analysis' / 'decay.png')))
zipbase = f'/content/{RESULTS_NAME}'
shutil.make_archive(zipbase, 'zip', OUT.parent, OUT.name)
print(f'{zipbase}.zip', os.path.getsize(f'{zipbase}.zip') // 1024, 'kB')
try:
    from google.colab import files
    files.download(f'{zipbase}.zip')
except Exception as e:
    print('descárguelo a mano:', e)